# 4V-FB manuscript experiments

This notebook is a thin Colab front end. The algorithms and experiment grids live in the Python package, so Colab and VS Code use the same source files. Run the smoke profile first; enable the full paper profile only after it succeeds.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = "https://github.com/YOUR_ACCOUNT/lp-lasso-4vfb-experiments.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if "YOUR_ACCOUNT" in REPO_URL:
        raise ValueError("Replace REPO_URL with your public or anonymous GitHub repository URL.")
    root = Path("/content/lp-lasso-4vfb-experiments")
    if not root.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(root)], check=True)
else:
    root = Path.cwd().resolve()
    if not (root / "pyproject.toml").is_file():
        raise FileNotFoundError("Start Jupyter from the repository root.")

os.chdir(root)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
print("Repository root:", root)

## Verify the frozen data used by the manuscript

This is fast and checks the saved CSV results against every quantitative claim encoded by the audit script.

In [ ]:
subprocess.run([sys.executable, "scripts/verify_manuscript_results.py"], check=True)

## Smoke run

The smoke profile exercises every module on a reduced grid. Its numbers are implementation checks, not paper results.

In [ ]:
smoke_output = root / "results" / "runs" / "colab_smoke"
subprocess.run([sys.executable, "run_all.py", "--profile", "smoke", "--experiments", "all", "--output", str(smoke_output)], check=True)

## Full paper run

Set `RUN_FULL=True` when ready. The command includes all grids and the one-million-step control, so it is intentionally not started by default.

In [ ]:
RUN_FULL = False
paper_output = root / "results" / "runs" / "colab_paper"
if RUN_FULL:
    subprocess.run([sys.executable, "run_all.py", "--profile", "paper", "--experiments", "all", "--output", str(paper_output)], check=True)
else:
    print("Full run skipped. Set RUN_FULL=True to execute it.")

## Download generated results

This packages the selected output directory, including raw CSV files, metadata, and checksums.

In [ ]:
import shutil
selected_output = paper_output if RUN_FULL else smoke_output
archive = Path(shutil.make_archive(str(selected_output), "zip", selected_output))
print("Created:", archive)
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))